In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

# Main repository directory
BASE_DIR = Path.cwd().parent

# Repository folders
OUTPUT_DIR = BASE_DIR / "outputs"
MODEL_DIR = BASE_DIR / "models"

# Create model folder if it does not exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", BASE_DIR)
print("Outputs folder exists:", OUTPUT_DIR.exists())
print("Models folder exists:", MODEL_DIR.exists())

Repository: C:\Users\Lenovo\OneDrive\Documents\StadioCare-performance-analysis
Outputs folder exists: True
Models folder exists: True


In [5]:

# Load processed feature matrices
X_train = joblib.load(
    OUTPUT_DIR / "X_train_processed.joblib"
)

X_test = joblib.load(
    OUTPUT_DIR / "X_test_processed.joblib"
)

# Load processed target variables
y_train = joblib.load(
    OUTPUT_DIR / "y_train_processed.joblib"
)

y_test = joblib.load(
    OUTPUT_DIR / "y_test_processed.joblib"
)

# Convert target variables to one-dimensional arrays
y_train = np.asarray(y_train).ravel()
y_test = np.asarray(y_test).ravel()

# Display dataset information
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

print("Training positive cases:", np.sum(y_train == 1))
print("Testing positive cases:", np.sum(y_test == 1))

Training features: (81613, 2419)
Testing features: (20153, 2419)
Training target: (81613,)
Testing target: (20153,)
Training positive cases: 9206
Testing positive cases: 2151


### Building and train the Logistic Regression model

In [7]:

logistic_model = LogisticRegression(
    C=1.0,
    solver="liblinear",
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

# Train the model
logistic_model.fit(X_train, y_train)

print("Logistic Regression training completed.")

Logistic Regression training completed.


### Predictions

In [8]:
# Predict the binary class
y_pred = logistic_model.predict(X_test)

# Predict probability of readmission within 30 days
y_prob = logistic_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully.")

print("First 10 predicted classes:")
print(y_pred[:10])

print("\nFirst 10 predicted probabilities:")
print(y_prob[:10])

Predictions generated successfully.
First 10 predicted classes:
[1 1 1 0 0 0 0 1 0 0]

First 10 predicted probabilities:
[0.56551383 0.55083976 0.5224403  0.14711565 0.38378378 0.35665643
 0.38229693 0.82063384 0.34023776 0.39499317]


### Evaluating the model

In [9]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, y_pred, zero_division=0
)

recall = recall_score(
    y_test, y_pred, zero_division=0
)

f1 = f1_score(
    y_test, y_pred, zero_division=0
)

roc_auc = roc_auc_score(
    y_test, y_prob
)

pr_auc = average_precision_score(
    y_test, y_prob
)

# Display results
results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC",
        "PR-AUC"
    ],
    "Score": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc,
        pr_auc
    ]
})

print(results.to_string(index=False))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Not readmitted within 30 days",
            "Readmitted within 30 days"
        ],
        zero_division=0
    )
)

   Metric    Score
 Accuracy 0.656676
Precision 0.166106
   Recall 0.551371
 F1-score 0.255301
  ROC-AUC 0.653494
   PR-AUC 0.201068

Classification Report:
                               precision    recall  f1-score   support

Not readmitted within 30 days       0.93      0.67      0.78     18002
    Readmitted within 30 days       0.17      0.55      0.26      2151

                     accuracy                           0.66     20153
                    macro avg       0.55      0.61      0.52     20153
                 weighted avg       0.84      0.66      0.72     20153



### Confusion matrix